## Steps

- Check where the -infs are coming from - check the sample genotypes
- Check if they are always for the same people
- Check if the NaNs are always for the same people across annotations

## Correlations
- Filter for unrelated european for getting the correlations

In [ ]:
import yaml
import polars as pl
import numpy as np
from anngeno import AnnGeno

from plotnine import *

In [ ]:
config_path = "/home/dnanexus/ukbgym/config_wgs_absplice2.yaml"

with open(config_path) as f:
    config = yaml.safe_load(f)

cov_list = config.get('covariates')

all_annotation_list = []
rare_variant_annotations_dict = config.get('rare_variant_annotations')
if rare_variant_annotations_dict:
    for category in rare_variant_annotations_dict.values():
        all_annotation_list.extend(category)

all_annotation_list = list(set(all_annotation_list))
len(all_annotation_list)


In [ ]:
s = pl.read_parquet('/home/dnanexus/absplice2_77_genes/ENSG00000143476.parquet')
# s['sample_id'].n_unique()
s

In [ ]:
s_inf = s.filter(
    (pl.col('sum').is_infinite()) |
    (pl.col('max').is_infinite()) |
    (pl.col('top2').is_infinite())
)

s_inf['annotation'].value_counts()

In [ ]:
pl.read_parquet('/home/dnanexus/absplice2_77_genes/ENSG00000116183.parquet').filter(
    (pl.col('sum').is_infinite()) |
    (pl.col('max').is_infinite()) |
    (pl.col('top2').is_infinite())
)['annotation'].value_counts()

In [ ]:
s.filter(
    (pl.col('sum').is_nan()) |
    (pl.col('max').is_nan()) |
    (pl.col('top2').is_nan())
)['annotation'].value_counts()

In [ ]:
# Remove rows where any of 'sum', 'max', or 'top2' is NaN or -inf
s_clean = s.filter(
    (~pl.col('sum').is_nan()) & (~pl.col('sum').is_infinite()) &
    (~pl.col('max').is_nan()) & (~pl.col('max').is_infinite()) &
    (~pl.col('top2').is_nan()) & (~pl.col('top2').is_infinite())
)

s_clean